## Source Notes Cleaner

Reads the raw source notes CSV from the Schwebel hard drive and produces a clean, one-row-per-article CSV for use on the website.

**Input:** `../_source_data/drschwebel_harddrive_data/articlemetadata_sourcenotes.csv`  
**Output:** `../_data/article_sourcenotes.csv`

### What this notebook does

1. Skips the first row (multi-line human-readable column descriptions) and the second row (internal short names)
2. Renames columns to clean snake_case headers
3. **Unfolds multi-article rows** — many rows have multiple article_ids crammed into one cell (newline-separated), with corresponding metadata also newline-separated in the same row. Each article_id becomes its own row, with shared fields (source_note, citation, publication) repeated and per-article fields (author, title, date, etc.) split out.
4. Normalises dates to `yyyy-mm-dd`
5. Cleans up `N/A` placeholder strings to empty
6. Strips stray whitespace throughout

In [1]:
import csv
import re
import pandas as pd
from datetime import datetime
from pathlib import Path

INFILE  = '../_source_data/drschwebel_harddrive_data/articlemetadata_sourcenotes.csv'
OUTFILE = '../_data/article_sourcenotes.csv'

# Target column names (in order, matching the 17 original columns)
NEW_COLS = [
    'number', 'publication', 'source_note', 'citation', 'article_id',
    'citation_format', 'author', 'title', 'volume_number', 'issue_number',
    'day', 'month', 'year', 'date', 'publisher', 'location', 'editor'
]

# Fields shared across all articles in a multi-article row
SINGLE_FIELDS = ['number', 'publication', 'source_note', 'citation']

# Fields that may have per-article values (newline-separated)
MULTI_FIELDS  = ['citation_format', 'author', 'title', 'volume_number',
                 'issue_number', 'day', 'month', 'year', 'date',
                 'publisher', 'location', 'editor']

In [2]:
# Read raw CSV — csv.reader handles quoted multi-line cells correctly
with open(INFILE, 'r', encoding='utf-8-sig', newline='') as f:
    reader = csv.reader(f)
    all_rows = list(reader)

# Row 0: multi-line human-readable description  → skip
# Row 1: internal short column names (id, publicationtitle, ...) → skip, use NEW_COLS
# Row 2+: data
data_rows = all_rows[2:]
print(f'Raw data rows (before unfolding): {len(data_rows)}')
print(f'Expected columns per row: {len(NEW_COLS)}')
print(f'Actual columns in first data row: {len(data_rows[0])}')

Raw data rows (before unfolding): 313
Expected columns per row: 17
Actual columns in first data row: 17


### Helper functions

In [3]:
def nl_split(val):
    """Split on newlines, strip each part, drop blanks."""
    if not val:
        return []
    parts = [p.strip() for p in str(val).split('\n')]
    return [p for p in parts if p]


def parse_date(s):
    """Normalise various date formats to yyyy-mm-dd."""
    if not s:
        return ''
    s = s.strip()
    if s in ('', 'N/A'):
        return ''

    # Already yyyy-mm-dd
    if re.match(r'^\d{4}-\d{2}-\d{2}$', s):
        return s

    # mm/dd/yyyy  (e.g. 5/4/1860)
    if re.match(r'^\d{1,2}/\d{1,2}/\d{4}$', s):
        try:
            return datetime.strptime(s, '%m/%d/%Y').strftime('%Y-%m-%d')
        except ValueError:
            pass

    # JavaScript Date.toString() style: "Sat Nov 01 1856 07:56:02 GMT+0300 (EAT)"
    m = re.match(r'\w+\s+(\w+)\s+(\d+)\s+(\d{4})', s)
    if m:
        try:
            return datetime.strptime(f'{m.group(1)} {m.group(2)} {m.group(3)}',
                                     '%b %d %Y').strftime('%Y-%m-%d')
        except ValueError:
            pass

    # yyyy-mm  → yyyy-mm-01
    if re.match(r'^\d{4}-\d{2}$', s):
        return s + '-01'

    # Year only (4 digits)
    if re.match(r'^\d{4}$', s):
        return s + '-01-01'

    return s  # return as-is if nothing matched


def clean_na(val):
    """Replace N/A, n/a, and bare dashes with empty string."""
    v = str(val).strip() if val else ''
    return '' if v.lower() in ('n/a', '----', '-', '') else v


def looks_like_multi_id(s):
    """Return True if a space-separated string looks like multiple article IDs.
    Article IDs are CamelCase/underscore tokens with no spaces inside them.
    Heuristic: if the string has 2+ space-separated tokens that each look
    like an article ID (only word chars + digits + underscore), it's multi."""
    tokens = s.split()
    if len(tokens) < 2:
        return False
    return all(re.match(r'^[A-Za-z0-9_\-\.]+$', t) for t in tokens)

### Unfold multi-article rows

Most multi-article rows have newline-separated values in every column that varies per article.  
A small number (e.g. Beloit Daily Call) use space separation instead.  
When a multi-value field has fewer values than article_ids, the last available value is repeated.

In [4]:
unfolded = []
multi_count = 0   # rows that were unfolded
problems   = []   # rows flagged for review

for row_idx, row in enumerate(data_rows):
    # Pad to 17 columns if the row is short
    while len(row) < len(NEW_COLS):
        row.append('')

    record = dict(zip(NEW_COLS, row[:len(NEW_COLS)]))

    # --- Determine article_id list ---
    raw_id = record['article_id'].strip()
    ids = nl_split(raw_id)

    # Fallback: space-separated IDs (Beloit-style edge case)
    if len(ids) == 1 and looks_like_multi_id(ids[0]):
        ids = [t.strip() for t in ids[0].split() if t.strip()]

    if not ids:
        # Row has no article_id — flag and skip
        problems.append((row_idx + 3, record.get('publication', ''), 'no article_id'))
        continue

    n = len(ids)

    # --- Split multi-value fields ---
    multi_vals = {}
    for field in MULTI_FIELDS:
        raw = record[field]
        parts = nl_split(raw)

        # For the space-separated Beloit edge case, also try splitting on 2+ spaces
        if len(parts) < n and raw and '  ' in raw:
            alt = [p.strip() for p in re.split(r'  +', raw) if p.strip()]
            if len(alt) == n:
                parts = alt

        multi_vals[field] = parts

    if n > 1:
        multi_count += 1

    # --- Emit one row per article_id ---
    for i, art_id in enumerate(ids):
        new_rec = {}

        # Shared fields
        for f in SINGLE_FIELDS:
            new_rec[f] = clean_na(record[f])

        new_rec['article_id'] = art_id

        # Per-article fields: use index i if available, else repeat last value
        for field in MULTI_FIELDS:
            parts = multi_vals[field]
            if i < len(parts):
                val = parts[i]
            elif parts:
                val = parts[-1]   # repeat last if not enough values
            else:
                val = ''
            new_rec[field] = clean_na(val)

        # Normalise date
        new_rec['date'] = parse_date(new_rec['date'])

        unfolded.append(new_rec)

print(f'Rows with multiple article_ids (unfolded): {multi_count}')
print(f'Total output rows:                          {len(unfolded)}')
if problems:
    print(f'\nRows flagged for review ({len(problems)}):')
    for p in problems:
        print(f'  CSV line {p[0]}: {p[1]} — {p[2]}')

Rows with multiple article_ids (unfolded): 64
Total output rows:                          439

Rows flagged for review (1):
  CSV line 243: Pioneer (RU) — no article_id


In [5]:
df = pd.DataFrame(unfolded, columns=NEW_COLS)

# Strip stray whitespace from all string columns
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda c: c.str.strip())

# Normalise remaining N/A strings that slipped through
NA_VALS = {'N/A', 'n/a', 'NA', '----', '-'}
df.replace(list(NA_VALS), '', inplace=True)

print(df.shape)
df.head(10)

(439, 17)


,number,publication,source_note,citation,article_id,citation_format,author,title,volume_number,issue_number,day,month,year,date,publisher,location,editor
0,2,Adams (PA) Sentinel and General Advertiser,<i>Adams Sentinel and General Advertiser</i> (...,"Charles H. Glatfelter, ""Adams County Votes for...",TheAdamsSentinel1853,newspaper-article,,"""A Female Crusoe.""","54,",2,7,November,1853,1853-11-07,Robert G. Harper,"Gettysburg, Pennsylvania",Robert G. Harper
1,3,Adventure Magazine,<i>Adventure Magazine</i> (1910-1971) was a pu...,"Patrick Scott Belk, ""<i>Adventure</i> Magazine...",Woodward1930,magazine-article,"Woodward, Arthur.","""A Female Robinson Crusoe.""",,,1,February,1930,1930-02-01,,"New York, New York",Albert A. Proctor
2,4,Air Time,<i>Air Time</i> (2004-14) was a publication th...,"Patricia Sauers, email message to Paige Kueste...",Sauers2013_AirTime,magazine-article,"Sauers, Patricia.","""Lost Indian Cave Found on San Nicolas Island.""","4,",2,,Summer,2013,2013-06-01,,,
3,12,Ancestors West,<i>Ancestors West</i> (1974-current) is the qu...,"""Ancestors West,"" <i>Santa Barbara County Gene...",Dittman2004,newspaper-newsletter-article,"Dittman, Richard H.","""Good Grief, Charley Brown!""","31,",1 & 2,,Fall/Winter,2004/2005,2004-09-01,,"Santa Barbara, California",
4,16,Art and Archaeology,<i>Art and Archaelogy</i> (1914-34) was an ill...,"<i>Art and Archaeology</i> 3 (1916): 2, access...",Bryan1930,magazine-article,"Bryan, Bruce.","""San Nicolas Island, Treasure House of the Anc...","29,",4,,April,1930,1930-04-01,,,
5,41,California Archaeology,<i>California Archaeology</i> (2009-current) i...,"""California Archaeology – The SCA Journal, <i>...",SchwartzVellanoweth2013,journal-article,"Schwartz, Steven J. and Rene L. Vellanoweth.","""Lone Woman's Cave Found on San Nicolas Island.""","5,",2,,December,2013,2013-12-01,,,
6,44,California Historical Society Quarterly,<i>California Historical Society Quarterly</i>...,"Carol Kammen and Amy H. Wilson, eds., <i>Encyc...",Thomson1953,journal-article,"Thomson, Virginia.","""Note on George Nidever: A ""Clean-Living and U...","32,",3,,,1953,1953-01-01,,"San Francisco, California",
7,45,California Territorial Quarterly,<i>California Territorial Quarterly</i> (1990-...,"Penny Anderson, ""A Short History of the <i>Dog...",Harrigan1998,journal-article,"Harrigan, John P.","""The Lone Woman of San Nicolas Island. ""","33,",33,,,1998,1998-01-01,,"Paradise, California",
8,67,New York Current Literature,<i>Current Literature</i> (1888-1925) was a mo...,"Frank Luther Mott, American Journalism: A Hist...",CurrentLiterature1892,newspaper-article,,"""Isle of Skulls.""","10,",1,,May,1892,1892-05-01,,"New York, New York",
9,68,Currents,<i>Currents</i> (2001-current) is the quarterl...,"""Currents,"" <i>U.S. Navy</i>, accessed Septemb...",Sauers2013_Currents,magazine-article,"Sauers, Patricia.","""Point Mugu’s Lead Archaeologist Makes a Signi...",,,,Fall,2013,2013-09-01,,,


### Diagnostics — inspect unfolded multi-article rows

In [6]:
# Publications that produced more than one output row (were unfolded)
pub_counts = df.groupby('publication')['article_id'].count()
multi_pub = pub_counts[pub_counts > 1].sort_values(ascending=False)
print('Publications with multiple article rows:')
print(multi_pub.to_string())

print(f'\nTotal unique article_ids: {df["article_id"].nunique()}')
print(f'Total rows: {len(df)}')

# Flag duplicate article_ids (same article_id appearing twice)
dups = df[df.duplicated('article_id', keep=False)]
if len(dups):
    print(f'\nWARNING: {len(dups)} rows have duplicate article_ids:')
    print(dups[['article_id', 'publication', 'date']].to_string())
else:
    print('\nNo duplicate article_ids — all IDs are unique. ✓')

Publications with multiple article rows:
publication
Los Angeles Times                                   13
San Francisco Chronicle                              8
Los Angeles Herald                                   8
Overland Monthly                                     8
San Diego Union                                      7
Santa Barbara (CA) Daily Press                       6
Rochester (NY) Democrat and Chronicle                5
Los Angeles Times Magazine                           5
Chicago Daily Tribune                                5
Galveston (TX) Daily News                            4
Cincinnati Enquirer                                  4
Horn Book Magazine                                   4
Oxnard (CA) Daily Courier                            4
New York Herald                                      3
Santa Barbara (CA) News-Press                        3
Masterkey                                            3
Los Angeles Tidings                                  3
Omaha (NE) D

In [7]:
# Spot-check a few known multi-article publications
spot_check = ['Overland Monthly', 'Cambria (PA) Freeman',
              'Los Angeles Herald', 'Beloit (KS) Daily Call',
              'Hutchings\' California Illustrated Magazine']

for pub in spot_check:
    subset = df[df['publication'].str.contains(pub.split('(')[0].strip(), na=False,
                                                case=False, regex=False)]
    if len(subset):
        print(f'\n--- {pub} ({len(subset)} rows) ---')
        print(subset[['article_id', 'author', 'date', 'title']].to_string(index=False))


--- Overland Monthly (8 rows) ---
    article_id                 author       date                                                                     title
   Walcott1872    Walcott, Josephine. 1872-08-01                                                             "Hona Maria."
      Dall1874            Dall, W. H. 1874-06-15                                                 "The Lords of the Isles."
Schumacher1875      Schumacher, Paul. 1875-10-01                                        "Some Remains of a Former People."
 McDougall1890  McDougall, William H. 1890-09-01                                                  "A Woman's Log of 1849."
   Kinsell1891   Kinsell, Martinette. 1891-01-01                                              "The Santa Barbara Islands."
     Yates1896 Yates, Lorenzo Gordin. 1896-05-01 "The Deserted Homes of a Lost People: The Santa Barbara Channel Islands."
      Paul1911        Paul, Oliver M. 1911-07-15                                                 "Female

### Known data fixes

These are errors in the source CSV corrected here before saving.

| Issue | Action | Reason |
|---|---|---|
| Philadelphia Public Ledger row contains two article_ids: `PublicLedger1847` (correct) and `ThePublicLedger1879` (erroneous) | Drop the `ThePublicLedger1879` Philadelphia row | `ThePublicLedger1879` belongs to the St. John's (CAN) Public Ledger, which has its own correct entry. The second ID was accidentally included in the multi-value Documents cell for Philadelphia. |
| Pioneer (RU) — no article_id in Documents column | Skipped entirely (flagged in diagnostics) | No article_id to key on; cannot be linked to site articles. |

In [8]:
# Fix: Philadelphia Public Ledger has two article_ids in the source CSV:
#   PublicLedger1847  — correct (1847 Boston Atlas direct reprint)
#   ThePublicLedger1879 — erroneous (belongs to St. John's (CAN) Public Ledger,
#                         which has its own separate correct entry)
# Drop the erroneous Philadelphia row rather than renaming it.
mask_drop = (
    (df['article_id'] == 'ThePublicLedger1879') &
    (df['publication'].str.contains('Philadelphia', na=False))
)
n_dropped = mask_drop.sum()
df = df[~mask_drop].reset_index(drop=True)
print(f"Dropped {n_dropped} erroneous row(s): Philadelphia Public Ledger / ThePublicLedger1879")

# Confirm no remaining duplicates
dups = df[df.duplicated('article_id', keep=False)]
if len(dups):
    print(f"WARNING: {len(dups)} duplicate article_ids remain:")
    print(dups[['article_id', 'publication']].to_string())
else:
    print("All article_ids are now unique. ✓")
print(f"Final row count: {len(df)}")

Dropped 1 erroneous row(s): Philadelphia Public Ledger / ThePublicLedger1879
All article_ids are now unique. ✓
Final row count: 438


### Strip HTML tags and save output

In [9]:
def strip_html(val):
    if not isinstance(val, str) or not val:
        return val
    return re.sub(r'<[^>]+>', '', val).strip()

str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda col: col.map(strip_html))

# Spot-check: confirm no remaining HTML tags in source_note or citation
html_in_note = df['source_note'].str.contains(r'<[^>]+>', regex=True, na=False).sum()
html_in_cite = df['citation'].str.contains(r'<[^>]+>', regex=True, na=False).sum()
print(f'Remaining HTML tags in source_note: {html_in_note}')
print(f'Remaining HTML tags in citation:    {html_in_cite}')

Remaining HTML tags in source_note: 0
Remaining HTML tags in citation:    0


In [10]:
df.to_csv(OUTFILE, index=False)
print(f'Saved {len(df)} rows to {OUTFILE}')
print(f'Columns: {list(df.columns)}')

Saved 438 rows to ../_data/article_sourcenotes.csv
Columns: ['number', 'publication', 'source_note', 'citation', 'article_id', 'citation_format', 'author', 'title', 'volume_number', 'issue_number', 'day', 'month', 'year', 'date', 'publisher', 'location', 'editor']
